# Recurrent layers, and the first model that beats the baseline

A single LSTM layer with 16 units — fewer parameters than the dense model that failed, and the first thing in this chapter that works.

**Runs on:** CPU — about 15 minutes (GPU: 3 minutes) &nbsp;·&nbsp; **Slides:** [Chapter 13 — Timeseries Forecasting](../../../course-web-slides/ch13/index.html) &nbsp;·&nbsp; **Section:** 03 — Recurrent neural networks

---

## A recurrent layer, written out

In [ ]:
import numpy as np

timesteps, input_features, output_features = 100, 32, 64
inputs = np.random.random((timesteps, input_features))
state_t = np.zeros((output_features,))

W = np.random.random((output_features, input_features))
U = np.random.random((output_features, output_features))
b = np.random.random((output_features,))

successive_outputs = []
for input_t in inputs:
    output_t = np.tanh(np.dot(W, input_t) + np.dot(U, state_t) + b)
    successive_outputs.append(output_t)
    state_t = output_t          # <- the recurrence
final_output_sequence = np.stack(successive_outputs, axis=0)
print(final_output_sequence.shape)

`state_t = output_t` is the entire idea. The output at each step depends on the input **and on everything that came before**, carried in a state vector. Chapter 15 will replace this loop with attention and get a large speed-up for exactly this reason: a loop cannot be parallelised.

## An LSTM on Jena

In [ ]:
import keras
from keras import layers
import matplotlib.pyplot as plt

NAIVE_MAE = 2.44
sequence_length, n_features = 120, 14

inputs = keras.Input(shape=(sequence_length, n_features))
x = layers.LSTM(16)(inputs)
outputs = layers.Dense(1)(x)
lstm_model = keras.Model(inputs, outputs)

lstm_model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
cb = [keras.callbacks.ModelCheckpoint("jena_lstm.keras", save_best_only=True)]
h_lstm = lstm_model.fit(train_dataset, epochs=10,
                        validation_data=val_dataset, callbacks=cb, verbose=2)

best = keras.models.load_model("jena_lstm.keras")
mae_lstm = best.evaluate(test_dataset, verbose=0)[1]
print(f"\nLSTM test MAE: {mae_lstm:.2f} degC   (baseline {NAIVE_MAE})")
print(f"parameters: {lstm_model.count_params():,}")

Expected output:

```
LSTM test MAE: 2.3x degC   (baseline 2.44)
parameters: 2,001
```

**Two thousand parameters**, against nearly 27,000 for the dense model that lost to doing nothing. Architecture, not capacity.

## The picture

In [ ]:
plt.figure(figsize=(8, 4.6))
plt.plot(h_lstm.history["mae"], lw=1, label="training")
plt.plot(h_lstm.history["val_mae"], lw=1.7, label="validation")
plt.axhline(NAIVE_MAE, color="k", ls="--", lw=1.4, label="naive baseline")
plt.xlabel("epoch"); plt.ylabel("MAE (degC)"); plt.legend()
plt.title("The first model in this chapter that beats doing nothing")
plt.show()

## LSTM against GRU

In [ ]:
def rnn(layer_cls, units=16, epochs=10, name=""):
    keras.utils.set_random_seed(0)
    i = keras.Input(shape=(sequence_length, n_features))
    x = layer_cls(units)(i)
    o = layers.Dense(1)(x)
    m = keras.Model(i, o)
    m.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    cb = [keras.callbacks.ModelCheckpoint(f"jena_{name}.keras",
                                          save_best_only=True)]
    h = m.fit(train_dataset, epochs=epochs, validation_data=val_dataset,
              callbacks=cb, verbose=0)
    mae = keras.models.load_model(f"jena_{name}.keras").evaluate(
        test_dataset, verbose=0)[1]
    print(f"{name:6s} {m.count_params():>7,} params   test MAE {mae:.3f}")
    return h, mae

h_g, mae_g = rnn(layers.GRU, name="gru")
h_s, mae_s = rnn(layers.SimpleRNN, name="simple")

**GRU** has three gates against LSTM's four — fewer parameters, usually comparable results, and it is what chapter 15 uses.

**SimpleRNN** is the loop from the first cell with no gates at all. It performs poorly on sequences of this length, and that is the vanishing-gradient problem: information from 120 steps ago has to survive 120 multiplications. Gates give it a path that does not.

## What the gates are for

In [ ]:
print("SimpleRNN:  state_t = tanh(W.x + U.state + b)")
print()
print("LSTM adds a separate carry track, and three gates that control it:")
print("  forget gate  -- what to drop from the carry")
print("  input gate   -- what to add to it")
print("  output gate  -- what of it to expose")
print()
print("The carry can pass through many steps ~unchanged, which is")
print("exactly the residual-connection idea from chapter 9, applied to time.")

## How far back does it actually look?

In [ ]:
import numpy as np

for L in [24, 48, 120, 240]:
    ds = keras.utils.timeseries_dataset_from_array(
        raw_data[:-delay], targets=temperature[delay:],
        sampling_rate=6, sequence_length=L, shuffle=True, batch_size=256,
        start_index=0, end_index=num_train_samples)
    vds = keras.utils.timeseries_dataset_from_array(
        raw_data[:-delay], targets=temperature[delay:],
        sampling_rate=6, sequence_length=L, shuffle=True, batch_size=256,
        start_index=num_train_samples,
        end_index=num_train_samples + num_val_samples)
    keras.utils.set_random_seed(0)
    i = keras.Input(shape=(L, n_features))
    m = keras.Model(i, layers.Dense(1)(layers.GRU(16)(i)))
    m.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    hh = m.fit(ds, epochs=6, validation_data=vds, verbose=0)
    print(f"history {L:3d} steps ({L//24} days): "
          f"best val MAE {min(hh.history['val_mae']):.3f}")

More history is not monotonically better. Beyond a few days the extra steps are mostly noise, and the model pays for them in training time and in gradient path length. **The window length is a hyperparameter**, and chapter 18's tuner would find it for you.

---

## What to take away

- A recurrent layer carries state forward: `state_t = output_t`.
- A 2,000-parameter LSTM beats a 27,000-parameter dense model here — prior, not capacity.
- Gates give information a path through time that does not vanish, the same idea as residual connections.
- Window length is a hyperparameter; longer is not automatically better.